# 4 — Birleşik Öncelik Skoru (Lead Priority Demo)

> **Yazılı karşılığı:** [`docs/4_priority_score.docx`](../docs/4_priority_score.docx) — formül seçimi, ağırlık gerekçesi, test seti tasarım uyumsuzluğunun bilinçli kabulü ve Phase 5 entegrasyon planı docx'te. Bu defter mevcut cache'leri (`artifacts/lead_scoring_lgbm.joblib`, `artifacts/sentiment_predictions/glm-4-5-air_test.parquet`, `artifacts/priority_metrics.json`) sade ekran tabloları + grafiklerle bir araya getirir; OpenRouter'a tekrar çağrı yapmaz ve hiçbir model yeniden eğitilmez.

## 1. Kurulum ve veri yükleme

`scripts/build_4_priority_docx.py` Phase 4 docx ve canonical metrikleri (`artifacts/priority_metrics.json`) üretir. Bu defter aynı zinciri (raw CSV → feature transformer → LGBM → P(conversion); sentiment cache → predicted_attitude; ikisinin weighted average'i) interaktif olarak takip eder. `compute_priority` ve `batch_compute_priority` `src/lead_priority/core/scoring/priority.py` altında — saf serving fonksiyonu, fit/eval kodu yok.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from lead_priority.core.features.constants import SEED
from lead_priority.core.features.pipeline import FeatureTransformer
from lead_priority.core.inference.lead_scoring import LeadScoringModel
from lead_priority.core.scoring.priority import batch_compute_priority, compute_priority
from lead_priority.core.scoring.sentiment_classes import SENTIMENT_CLASSES, SENTIMENT_SCORE_MAP
from lead_priority.settings import REPO_ROOT, get_settings

RAW_CSV = REPO_ROOT / 'data' / 'Lead Scoring.csv'
INTERACTIONS = REPO_ROOT / 'data' / 'synthetic' / 'interactions.parquet'
PREDICTIONS = REPO_ROOT / 'artifacts' / 'sentiment_predictions' / 'glm-4-5-air_test.parquet'
FEATURE_PIPELINE = REPO_ROOT / 'artifacts' / 'feature_pipeline.joblib'
LGBM = REPO_ROOT / 'artifacts' / 'lead_scoring_lgbm.joblib'
PRIORITY_METRICS = REPO_ROOT / 'artifacts' / 'priority_metrics.json'

raw = pd.read_csv(RAW_CSV)
interactions = pd.read_parquet(INTERACTIONS)
predictions = pd.read_parquet(PREDICTIONS)
metrics = json.loads(PRIORITY_METRICS.read_text())

print(f'raw leads: {len(raw):,} | interactions: {len(interactions):,} | sentiment cache: {len(predictions):,}')
print(f'priority_metrics.json schema={metrics["schema_version"]}, n_full={metrics["n_full"]}, n_clean={metrics["n_clean"]}')

## 2. Phase 2 ve Phase 3 test split'lerini yeniden üret

İki bağımsız split var (docx §6'da gerekçeli):

- **Phase 2 LGBM**: `test_size=0.20`, `stratify=Converted`, `seed=42` → 1.848 lead test.
- **Phase 3 sentiment**: `test_size=0.10`, `stratify=attitude+language`, `seed=42` → 924 lead test.

Aynı seed ama farklı stratify anahtarı + farklı `test_size` ⇒ farklı partisyonlar. Phase 4 demo'su 924 sentiment-test lead'i üzerinde çalışır; bunların hangileri LGBM'in train/val/test setlerinde olduğunu aşağıda işaretliyoruz.

In [ ]:
# Phase 2 split
prospect_ids = raw['Prospect ID'].to_numpy()
y = raw['Converted'].to_numpy().astype(int)
pids_trainval, pids_test, y_trainval, y_test = train_test_split(
    prospect_ids, y, test_size=0.20, stratify=y, random_state=SEED,
)
pids_train, pids_val, _, _ = train_test_split(
    pids_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED,
)
lgbm_split = {
    'train': set(map(str, pids_train)),
    'val':   set(map(str, pids_val)),
    'test':  set(map(str, pids_test)),
}
print({k: len(v) for k, v in lgbm_split.items()})

# Phase 3 split (sanity — should give 924, same as cache)
strata = interactions['attitude'].astype(str) + '_' + interactions['language'].astype(str)
_, sentiment_test_part = train_test_split(
    interactions, test_size=0.10, random_state=SEED, stratify=strata, shuffle=True,
)
sentiment_test_ids = set(sentiment_test_part['lead_id'].astype(str))
cache_ids = set(predictions['lead_id'].astype(str))
print(f'Phase 3 reproduced test set size: {len(sentiment_test_ids)}; cache size: {len(cache_ids)}; match: {sentiment_test_ids == cache_ids}')

## 3. P(conversion) hesabı + birleşik öncelik

924 sentiment-test lead'inin tabular feature'larını `Lead Scoring.csv`'den çekip Phase 2 transformer + LGBM ile P(conversion)'a çeviriyoruz. Sentiment skorunu `predicted_attitude` üzerinden `SENTIMENT_SCORE_MAP` ile çıkarıp `batch_compute_priority` ile birleştiriyoruz.

In [ ]:
transformer = FeatureTransformer.load(FEATURE_PIPELINE)
lgbm = LeadScoringModel.load(LGBM)
settings = get_settings()

raw_subset = raw[raw['Prospect ID'].astype(str).isin(cache_ids)].reset_index(drop=True)
x = transformer.transform(raw_subset)
p_conv = lgbm.predict_proba(x)

base = raw_subset[['Prospect ID', 'Converted']].rename(columns={'Prospect ID': 'prospect_id', 'Converted': 'converted'})
base['p_conversion'] = p_conv
pred_view = predictions[['lead_id', 'language', 'predicted_attitude', 'true_attitude']].rename(columns={'lead_id': 'prospect_id'})
merged = base.merge(pred_view, on='prospect_id', how='inner')
merged['in_lgbm_train'] = merged['prospect_id'].astype(str).isin(lgbm_split['train'])
merged['in_lgbm_val']   = merged['prospect_id'].astype(str).isin(lgbm_split['val'])
merged['in_lgbm_test']  = merged['prospect_id'].astype(str).isin(lgbm_split['test'])
merged['priority'] = batch_compute_priority(
    merged['p_conversion'].to_numpy(),
    merged['predicted_attitude'].astype(str).tolist(),
)
merged.head()

### Sanity — `compute_priority` skaler hali

Vektörize edilmiş çağrının her satırının skaler çağrıyla aynı sonucu verdiğini kontrol edelim. Ayrıca varsayılan ağırlıklar `Settings`'ten okunduğu için bir kez de orayı doğrulayalım.

In [ ]:
scalar_check = [
    compute_priority(p, a)
    for p, a in zip(merged['p_conversion'].head().to_list(), merged['predicted_attitude'].head().to_list())
]
np.testing.assert_allclose(scalar_check, merged['priority'].head().to_numpy())
print(f'Settings: w_conv={settings.priority_weight_conversion}, w_sent={settings.priority_weight_sentiment}')
print(f'Skaler == vektörize: OK')
print(f'Örnek: compute_priority(0.5, "objection") = {compute_priority(0.5, "objection"):.4f} (beklenen: 0.5600)')

## 4. Leakage tablosu — sentiment_test ∩ LGBM split

924 sentiment-test lead'inin LGBM split'lerinde dağılımı (docx §6.1). Eğitim/val'de yer alan lead'lerde P(conversion) optimistik; tüm dürüst operasyonel iddialar **temiz kesişim** (`in_lgbm_test=True`) üzerinden gider.

In [ ]:
leak = pd.DataFrame({
    'kuyu': ['LGBM train', 'LGBM val', 'LGBM test (temiz kesişim)', 'Toplam (924)'],
    'lead sayısı': [
        int(merged['in_lgbm_train'].sum()),
        int(merged['in_lgbm_val'].sum()),
        int(merged['in_lgbm_test'].sum()),
        len(merged),
    ],
})
leak['oran'] = (leak['lead sayısı'] / len(merged)).map(lambda x: f'{x:.1%}')
leak

## 5. Top-N capture — temiz 190 alt kümede (operasyonel)

Priority sıralı ilk N lead'in gerçek `Converted`'lerin kaçını yakaladığı. Base rate, alt kümedeki Converted oranı; lift = capture / base rate.

In [ ]:
clean = merged[merged['in_lgbm_test']].copy()
base_rate = clean['converted'].mean()

rows = []
for n in (10, 25, 50, 100):
    top = clean.nlargest(n, 'priority')
    captured = int(top['converted'].sum())
    capture_rate = captured / n
    rows.append({
        'N': n,
        'yakalanan Converted': captured,
        'capture oranı': f'{capture_rate:.2%}',
        'lift': f'{capture_rate / base_rate:.2f}×',
    })
print(f'Temiz alt küme: {len(clean)} lead, base rate {base_rate:.2%}')
pd.DataFrame(rows)

### Cumulative gain — temiz 190

Diyagonal kesikli çizgi rastgele sıralama; mavi eğri priority sıralı yakalama. Eğrinin diyagonalin ne kadar üstünde olduğu sıralamanın değerini gösterir.

In [ ]:
ranked = clean.sort_values('priority', ascending=False).reset_index(drop=True)
total_converted = int(ranked['converted'].sum())
n_total = len(ranked)
cum_converted = ranked['converted'].cumsum().to_numpy() / max(total_converted, 1)
fraction_called = (np.arange(n_total) + 1) / n_total

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(fraction_called, cum_converted, color='tab:blue', linewidth=2, label='Priority sıralı')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Rastgele baseline')
ax.set_xlabel('Aranan lead oranı (top-N)')
ax.set_ylabel('Yakalanan Converted oranı')
ax.set_title(f'Cumulative gain — temiz {n_total} lead, {total_converted} Converted')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.show()

## 6. Ağırlık duyarlılığı — w_conv sweep

Default `w_conv = 0.6` seçimini valide eder: w_conv ∈ {0.4, 0.5, 0.6, 0.7, 0.8} taramasında top-50 capture nasıl değişiyor. Düz çizgi → sinyaller uyumlu, seçim kritik değil; dik eğim → seçim önemli.

In [ ]:
sweep = []
for w_conv in (0.4, 0.5, 0.6, 0.7, 0.8):
    w_sent = 1.0 - w_conv
    scores = batch_compute_priority(
        clean['p_conversion'].to_numpy(),
        clean['predicted_attitude'].astype(str).tolist(),
        weight_conversion=w_conv,
        weight_sentiment=w_sent,
    )
    top = clean.assign(priority=scores).nlargest(50, 'priority')
    capture = top['converted'].mean()
    sweep.append({'w_conv': w_conv, 'w_sent': w_sent, 'capture@50': f'{capture:.2%}', 'lift': f'{capture / base_rate:.2f}×'})
pd.DataFrame(sweep)

## 7. Priority dağılımı + sınıf-bazlı bant (924 illüstratif)

Bu iki figür **leakage içerir** — ~561 lead LGBM eğitim setinde, P(conversion) skorları yapay olarak yukarı çekik. Operasyonel iddialar §5–§6'dan; bu görseller formülün davranışını anlatmak için.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax0 = axes[0]
bins = np.linspace(0, 1, 31)
ax0.hist(merged['priority'], bins=bins, alpha=0.55, label='Tüm 924 (illüstratif)', color='tab:blue')
ax0.hist(clean['priority'], bins=bins, alpha=0.85, label=f'{len(clean)} temiz kesişim', color='tab:orange')
ax0.set_xlabel('Priority skoru')
ax0.set_ylabel('Lead sayısı')
ax0.set_title('Priority dağılımı')
ax0.legend()
ax0.grid(alpha=0.3, axis='y')

ax1 = axes[1]
groups = [merged[merged['predicted_attitude'] == cls]['priority'].to_numpy() for cls in SENTIMENT_CLASSES]
bp = ax1.boxplot(groups, tick_labels=list(SENTIMENT_CLASSES), patch_artist=True, medianprops={'color': 'black'})
for patch, color in zip(bp['boxes'], ['tab:green', 'tab:red', 'tab:gray', 'tab:purple']):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
ax1.set_ylabel('Priority skoru')
ax1.set_title('Tahmin edilen sentiment × priority bandı (n=924, illüstratif)')
ax1.set_ylim(0, 1.05)
ax1.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Top-20 öncelikli lead listesi — temiz kesişim

Reviewer'in 'priority sıralaması gerçekten işliyor mu?' sorusuna doğrudan cevap. Son sütundaki `Converted` 1 ise model satış ekibini doğru lead'e yönlendirdi. Top-20'nin tamamına yakını 1 olmalı (capture oranı tabloda zaten kanıtlı).

In [ ]:
top20 = clean.nlargest(20, 'priority')[[
    'prospect_id', 'language', 'predicted_attitude', 'p_conversion', 'priority', 'converted',
]].copy()
top20['p_conversion'] = top20['p_conversion'].map(lambda v: f'{v:.4f}')
top20['priority'] = top20['priority'].map(lambda v: f'{v:.4f}')
top20.reset_index(drop=True)

## 9. Phase 5 (FastAPI) wiring

Aşağıdaki örnek tek bir HTTP request'in `compute_priority` zincirinden nasıl geçeceğini gösterir. Phase 5 endpoint'i bu dört çağrıyı (`transformer.transform`, `lgbm.predict_proba`, `sentiment.predict`, `compute_priority`) tek POST handler'ı içinde toplayacak. Sentiment çağrısı OpenRouter'a gider; timeout veya 429 durumunda fallback `attitude='neutral'` kullanmak makul bir tercih (priority sentiment skoru orta-düşük; satış ekibi yine de listeyi alır).

Burada OpenRouter'a çağrı **yapmıyoruz** — Phase 3 cache'inden ilk lead'i alıp sentiment çağrısının ne döneceğini simüle ediyoruz.

In [ ]:
example = clean.iloc[0]
example_pid = example['prospect_id']
example_raw = raw_subset[raw_subset['Prospect ID'] == example_pid]

# Phase 5 serving zincirinin notebook eşdeğeri
x_one = transformer.transform(example_raw)
p_one = float(lgbm.predict_proba(x_one)[0])
attitude_one = str(example['predicted_attitude'])  # Phase 5'te: sentiment.predict(interaction_text)
priority_one = compute_priority(p_one, attitude_one)

print(f'Prospect ID:        {example_pid}')
print(f'P(conversion):      {p_one:.4f}')
print(f'Predicted attitude: {attitude_one}')
print(f'Sentiment score:    {SENTIMENT_SCORE_MAP[attitude_one]:.2f}')
print(f'Priority:           {priority_one:.4f}')
print(f'Ground truth Converted: {int(example["converted"])}')

---

**Sonraki adım:** Phase 5 — bu zinciri FastAPI endpoint'ine taşı, OpenRouter'ı gerçek istek başına çağır. Test stratejisi: `respx`-mocked integration test + manuel curl smoke. Detaylar `docs/4_priority_score.docx` §10.